# RF-DETR instance segmentation (COCO v2)

Train `RFDETRSegSmall` on Roboflow-exported COCO. Activate conda env **`projeto_placentas`** before running.

**4GB VRAM:** defaults use `batch_size=1`, `grad_accum_steps=16`, `gradient_checkpointing=True`, `resolution=560`. If CUDA OOM, lower `resolution` (must stay divisible by 14).

Run cells **top to bottom** (paths cell must run before train and inference).

In [1]:
import torch
from importlib.metadata import version, PackageNotFoundError
from rfdetr import RFDETRSegSmall

try:
    _rf_ver = version("rfdetr")
except PackageNotFoundError:
    _rf_ver = "unknown"
print(f"rfdetr: {_rf_ver}")
print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")

d:\miniconda3\envs\projeto_placentas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rfdetr: 1.6.2
torch: 2.5.1+cu121
cuda available: True
device: NVIDIA GeForce GTX 1650 SUPER


In [2]:
from pathlib import Path

REPO_ROOT = Path(r"D:\projeto_placentas_clayton\dev\projeto-placentas").resolve()
V2_RF_DETR = REPO_ROOT / "v2_rf_detr"
DATASET_DIR = Path(r"D:\projeto_placentas_clayton\dataset_v2_coco_rf_detr").resolve()

OUTPUT_RUN = "seg_small_v1"
OUTPUT_DIR = (V2_RF_DETR / "runs" / OUTPUT_RUN).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATASET_DIR:", DATASET_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

DATASET_DIR: D:\projeto_placentas_clayton\dataset_v2_coco_rf_detr
OUTPUT_DIR: D:\projeto_placentas_clayton\dev\projeto-placentas\v2_rf_detr\runs\seg_small_v1


In [3]:
import json
from collections import Counter

def summarize_split(split: str) -> None:
    p = DATASET_DIR / split / "_annotations.coco.json"
    if not p.is_file():
        print(f"missing: {p}")
        return
    data = json.loads(p.read_text(encoding="utf-8"))
    cats = {c["id"]: c["name"] for c in data.get("categories", [])}
    anns = data.get("annotations", [])
    seg_n = sum(1 for a in anns if a.get("segmentation"))
    cat_ids = Counter(a.get("category_id") for a in anns)
    print(f"[{split}] images: {len(data.get('images', []))}  annotations: {len(anns)}  with segmentation: {seg_n}")
    print(f"[{split}] category_id counts: {dict(cat_ids)}")
    print(f"[{split}] categories: {cats}")

summarize_split("train")
summarize_split("valid")

[train] images: 153  annotations: 4753  with segmentation: 4752
[train] category_id counts: {1: 4753}
[train] categories: {0: 'objects', 1: 'microcotiledone'}
[valid] images: 27  annotations: 852  with segmentation: 852
[valid] category_id counts: {1: 852}
[valid] categories: {0: 'objects', 1: 'microcotiledone'}


In [4]:
model = RFDETRSegSmall()

model.train(
    dataset_dir=str(DATASET_DIR),
    output_dir=str(OUTPUT_DIR),
    epochs=120,
    batch_size=1,
    grad_accum_steps=16,
    lr=1e-4,
    gradient_checkpointing=True,
    resolution=560,
    early_stopping=True,
    early_stopping_patience=30,
    progress_bar=True,
    seed=0,
)

[2026-03-28 18:23:41] [INFO] rf-detr - File rf-detr-seg-small.pt already exists with correct MD5 hash.


[2026-03-28 18:23:41] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-03-28 18:23:41] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-03-28 18:23:41] [INFO] rf-detr - File rf-detr-seg-small.pt already exists with correct MD5 hash.


[2026-03-28 18:23:51] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-03-28 18:23:51] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-03-28 18:23:51] [INFO] rf-detr - File rf-detr-seg-small.pt already exists with correct MD5 hash.


...

Val — Overall Metrics                               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃     segm mAP    ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │ 50:95  │   50   │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.7932 │ 0.9147 │ 0.8388 │ 0.8597 │ 0.8924 │ 0.8988 │ 0.8862 │ 0.7630 │ 0.9211 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┘

                       Val — Per-class Metrics                       
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Class           ┃ AP 50:95 ┃     AR ┃     F1 ┃ Precision ┃ Recall ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ microcotiledone │   0.7932 │ 0.8597 │ 0.8924 │    0.8988 │ 0.8862 │
└─────────────────┴──────────┴────────┴────────┴───────────┴────────┘

Epoch 46: 100%|██████████| 153/153 [03:33<00:00,  0.72it/s, train/lr=0.0001, train/lr_min=3.23e-6, train/lr_max=0.0001, val/loss=6.760, val/mAP_50_95=0.793, val/mAP_50=0.915, val/ema_mAP_50_95=0.801, val/F1=0.892, train/loss=5.450]

Monitored metric __rfdetr_effective_map__ did not improve in the last 30 records. Best score: 0.808. Signaling Trainer to stop.


Epoch 46: 100%|██████████| 153/153 [03:39<00:00,  0.70it/s, train/lr=0.0001, train/lr_min=3.23e-6, train/lr_max=0.0001, val/loss=6.760, val/mAP_50_95=0.793, val/mAP_50=0.915, val/ema_mAP_50_95=0.801, val/F1=0.892, train/loss=5.420]
[2026-03-28 21:24:04] [INFO] rf-detr - Best total checkpoint saved from regular (regular=0.8081, ema=0.8079)


In [7]:
from pathlib import Path
from rfdetr import RFDETRSegSmall

OUTPUT_DIR = Path(r"D:\projeto_placentas_clayton\dev\projeto-placentas\v2_rf_detr\runs\seg_small_v1").resolve()
DATASET_DIR = Path(r"D:\projeto_placentas_clayton\dataset_v2_coco_rf_detr").resolve()

ckpt = OUTPUT_DIR / "checkpoint_best_total.pth"
assert ckpt.is_file(), f"missing checkpoint: {ckpt} — run training cell first"

infer_model = RFDETRSegSmall(pretrain_weights=str(ckpt))

valid_imgs = sorted((DATASET_DIR / "valid").glob("*.jpg"))
if not valid_imgs:
    valid_imgs = sorted((DATASET_DIR / "valid").glob("*.png"))
assert valid_imgs, "no jpg/png in valid/"

sample = valid_imgs[0]
detections = infer_model.predict(str(sample), threshold=0.5)
print("sample:", sample.name)
print(detections)

[2026-03-30 21:39:18] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-03-30 21:39:18] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-03-30 21:39:22] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().
`use_return_dict` is deprecated! Use `return_dict` instead!


sample: ROSILHA-M-B_003_jpg.rf.863a24cebe30a134937348de5cbcaa87.jpg
Detections(xyxy=array([[ 1.53435379e+02, -2.62222290e-01,  2.93127350e+02,
         2.07700958e+02],
       [ 1.49295568e+01,  1.65860580e+02,  1.58001694e+02,
         2.79225983e+02],
       [ 9.37325134e+01,  2.76210876e+02,  1.92140076e+02,
         3.54744263e+02],
       [ 2.90781586e+02, -7.28063583e-01,  4.47627716e+02,
         1.96830627e+02],
       [ 4.50275513e+02, -3.83005142e-01,  5.83812500e+02,
         1.84177216e+02],
       [ 3.11546326e-01,  3.07515991e+02,  8.52101822e+01,
         3.91211273e+02],
       [ 5.41188660e+02,  4.74656799e+02,  5.92676392e+02,
         5.15623230e+02],
       [ 3.89480347e+02,  1.24965134e+02,  4.46802063e+02,
         1.94998352e+02],
       [ 3.35158813e+02,  1.52360901e+02,  3.84799530e+02,
         2.31628784e+02],
       [ 6.03294373e-02,  1.07045174e-01,  1.29572464e+02,
         1.22521591e+02],
       [ 1.89579483e+02,  2.33180466e+02,  5.61032593e+02,
       

## Post-training validation (RF-DETR)

Mirrors the YOLO validation style with RF-DETR-compatible checks:
- confidence sweep to find best F1 operating point
- per-image merged-mask IoU and optional visual overlays
- per-instance TP/FP/FN report with area error statistics

In [1]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from rfdetr import RFDETRSegSmall

# Paths (reuse if already defined above)
if "DATASET_DIR" not in globals():
    DATASET_DIR = Path(r"D:\projeto_placentas_clayton\dataset_v2_coco_rf_detr").resolve()
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path(r"D:\projeto_placentas_clayton\dev\projeto-placentas\v2_rf_detr\runs\seg_small_v1").resolve()

CKPT_PATH = OUTPUT_DIR / "checkpoint_best_total.pth"
assert CKPT_PATH.is_file(), f"missing checkpoint: {CKPT_PATH}"

ANN_PATH = DATASET_DIR / "valid" / "_annotations.coco.json"
assert ANN_PATH.is_file(), f"missing COCO annotations: {ANN_PATH}"

with ANN_PATH.open("r", encoding="utf-8") as f:
    coco = json.load(f)

images = coco.get("images", [])
annotations = coco.get("annotations", [])

img_by_id = {int(im["id"]): im for im in images}
anns_by_img = {}
for ann in annotations:
    anns_by_img.setdefault(int(ann["image_id"]), []).append(ann)

# Keep only images that physically exist in valid/
valid_img_paths = []
for im in images:
    p = DATASET_DIR / "valid" / im["file_name"]
    if p.is_file():
        valid_img_paths.append(p)

assert valid_img_paths, "No validation images found in DATASET_DIR/valid"

print(f"Validation images found: {len(valid_img_paths)}")
print(f"Validation annotations: {len(annotations)}")


def ann_to_mask(ann, height, width):
    """Convert COCO annotation segmentation to binary mask."""
    seg = ann.get("segmentation", None)
    if seg is None:
        return np.zeros((height, width), dtype=np.uint8)

    # Polygon format
    if isinstance(seg, list):
        mask = np.zeros((height, width), dtype=np.uint8)
        for poly in seg:
            if not poly or len(poly) < 6:
                continue
            pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
            cv2.fillPoly(mask, [pts.astype(np.int32)], 1)
        return mask

    # RLE format (fallback if present)
    if isinstance(seg, dict):
        try:
            from pycocotools import mask as mask_utils

            return mask_utils.decode(seg).astype(np.uint8)
        except Exception:
            return np.zeros((height, width), dtype=np.uint8)

    return np.zeros((height, width), dtype=np.uint8)


def get_gt_masks_for_image(img_id):
    im = img_by_id[int(img_id)]
    h, w = int(im["height"]), int(im["width"])
    masks = [ann_to_mask(a, h, w) for a in anns_by_img.get(int(img_id), [])]
    return masks, h, w


def get_pred_masks_and_conf(det):
    masks = []
    confs = []

    pred_mask = getattr(det, "mask", None)
    pred_conf = getattr(det, "confidence", None)

    if pred_mask is not None:
        arr = np.asarray(pred_mask)
        if arr.ndim == 2:
            arr = arr[None, ...]
        for m in arr:
            masks.append((m > 0).astype(np.uint8))

    if pred_conf is not None:
        confs = list(np.asarray(pred_conf, dtype=np.float32))

    # If confidence array is missing/mismatched, fallback to all ones
    if len(confs) != len(masks):
        confs = [1.0] * len(masks)

    return masks, confs


def iou_binary(a, b):
    inter = np.logical_and(a > 0, b > 0).sum()
    union = np.logical_or(a > 0, b > 0).sum()
    return float(inter / union) if union > 0 else 0.0


def greedy_match(pred_masks, gt_masks, iou_thr=0.5):
    n_p, n_g = len(pred_masks), len(gt_masks)
    if n_p == 0 or n_g == 0:
        return []

    mat = np.zeros((n_p, n_g), dtype=np.float32)
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            mat[i, j] = iou_binary(pm, gm)

    matched_p, matched_g = set(), set()
    pairs = []

    order = np.argsort(-mat, axis=None)
    pi, gi = np.unravel_index(order, mat.shape)

    for p_idx, g_idx in zip(pi, gi):
        if p_idx in matched_p or g_idx in matched_g:
            continue
        val = float(mat[p_idx, g_idx])
        if val < iou_thr:
            break
        pairs.append((p_idx, g_idx, val))
        matched_p.add(p_idx)
        matched_g.add(g_idx)

    return pairs


infer_model = RFDETRSegSmall(pretrain_weights=str(CKPT_PATH))
infer_model.optimize_for_inference()
print("RF-DETR loaded and optimized for inference.")

d:\miniconda3\envs\projeto_placentas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-04-04 14:57:09] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-04 14:57:09] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


Validation images found: 27
Validation annotations: 852


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
d:\miniconda3\envs\projeto_placentas\lib\site-packages\rfdetr\models\backbone\dinov2.py:212: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert x.shape[2] % block_size == 0 and x.shape[3] % block_size == 0, (
`use_return_dict` is deprecated! Use `return_dict` instead!
d:\miniconda3\envs\projeto_placentas\lib\site-packages\rfdetr\models\backbone\dinov2_with_windowed_attn.py:313: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_channels != self

RF-DETR loaded and optimized for inference.


In [2]:
# 1) Confidence sweep (YOLO-like "optimal confidence" and F1)

IOU_THRESHOLD = 0.5
CONF_GRID = np.round(np.linspace(0.10, 0.90, 17), 3)

sweep_rows = []

for conf_thr in CONF_GRID:
    total_tp = total_fp = total_fn = 0
    tp_ious = []
    image_ious = []

    for img_path in valid_img_paths:
        img_name = img_path.name

        # Find image_id from COCO by filename
        img_id = None
        for k, im in img_by_id.items():
            if im.get("file_name") == img_name:
                img_id = int(k)
                break
        if img_id is None:
            continue

        gt_masks, h, w = get_gt_masks_for_image(img_id)

        det = infer_model.predict(str(img_path), threshold=float(conf_thr))
        pred_masks, pred_confs = get_pred_masks_and_conf(det)

        # Keep explicit thresholding by confidence for safety
        pred_masks = [m for m, c in zip(pred_masks, pred_confs) if c >= conf_thr]

        # Resize predictions if needed to GT shape
        resized_preds = []
        for pm in pred_masks:
            if pm.shape != (h, w):
                pm = cv2.resize(pm.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
            resized_preds.append((pm > 0).astype(np.uint8))
        pred_masks = resized_preds

        pairs = greedy_match(pred_masks, gt_masks, iou_thr=IOU_THRESHOLD)

        tp = len(pairs)
        fp = len(pred_masks) - tp
        fn = len(gt_masks) - tp

        total_tp += tp
        total_fp += fp
        total_fn += fn

        tp_ious.extend([p[2] for p in pairs])

        gt_union = np.zeros((h, w), dtype=np.uint8)
        for gm in gt_masks:
            gt_union = np.logical_or(gt_union, gm).astype(np.uint8)

        pred_union = np.zeros((h, w), dtype=np.uint8)
        for pm in pred_masks:
            pred_union = np.logical_or(pred_union, pm).astype(np.uint8)

        image_ious.append(iou_binary(gt_union, pred_union))

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    sweep_rows.append(
        {
            "conf": float(conf_thr),
            "tp": int(total_tp),
            "fp": int(total_fp),
            "fn": int(total_fn),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "mean_iou_tp": float(np.mean(tp_ious)) if tp_ious else 0.0,
            "mean_iou_image_union": float(np.mean(image_ious)) if image_ious else 0.0,
        }
    )

sweep_df = pd.DataFrame(sweep_rows).sort_values("conf").reset_index(drop=True)
BEST_ROW = sweep_df.loc[sweep_df["f1"].idxmax()]
BEST_CONF = float(BEST_ROW["conf"])

print("\n" + "=" * 60)
print("RF-DETR VALIDATION (confidence sweep)")
print("=" * 60)
print(f"Best confidence:      {BEST_CONF:.3f}")
print(f"Peak F1:              {BEST_ROW['f1']:.4f}")
print(f"Precision @best:      {BEST_ROW['precision']:.4f}")
print(f"Recall @best:         {BEST_ROW['recall']:.4f}")
print(f"TP/FP/FN @best:       {int(BEST_ROW['tp'])}/{int(BEST_ROW['fp'])}/{int(BEST_ROW['fn'])}")
print(f"Mean IoU (TP pairs):  {BEST_ROW['mean_iou_tp']:.4f}")
print(f"Mean IoU (union/img): {BEST_ROW['mean_iou_image_union']:.4f}")
print("=" * 60)

SWEEP_CSV = OUTPUT_DIR / "validation_conf_sweep_rfdetr.csv"
sweep_df.to_csv(SWEEP_CSV, index=False)
print(f"Sweep table saved to: {SWEEP_CSV}")

display(sweep_df)


RF-DETR VALIDATION (confidence sweep)
Best confidence:      0.500
Peak F1:              0.8955
Precision @best:      0.9113
Recall @best:         0.8803
TP/FP/FN @best:       750/73/102
Mean IoU (TP pairs):  0.9020
Mean IoU (union/img): 0.8847
Sweep table saved to: D:\projeto_placentas_clayton\dev\projeto-placentas\v2_rf_detr\runs\seg_small_v1\validation_conf_sweep_rfdetr.csv


,conf,tp,fp,fn,precision,recall,f1,mean_iou_tp,mean_iou_image_union
0,0.10,818,689,34,0.542800,0.960094,0.693514,0.908161,0.936042
1,0.15,816,507,36,0.616780,0.957746,0.750345,0.908201,0.936404
2,0.20,812,402,40,0.668863,0.953052,0.786060,0.907665,0.937087
3,0.25,808,326,44,0.712522,0.948357,0.813696,0.906969,0.936953
4,0.30,805,253,47,0.760870,0.944836,0.842932,0.905591,0.936550
5,0.35,796,200,56,0.799197,0.934272,0.861472,0.905553,0.936429
6,0.40,785,149,67,0.840471,0.921362,0.879059,0.905268,0.932261
7,0.45,770,99,82,0.886076,0.903756,0.894829,0.902168,0.911527
8,0.50,750,73,102,0.911300,0.880282,0.895522,0.902001,0.884661
9,0.55,726,52,126,0.933162,0.852113,0.890798,0.902876,0.840435


In [3]:
# 2) Per-image merged-mask IoU + visual overlays (GT=green, AI=red, overlap=yellow)

import matplotlib.pyplot as plt

VIZ_DIR = OUTPUT_DIR / "validation_iou_viz_rfdetr"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

per_image_rows = []

for img_path in valid_img_paths:
    img_name = img_path.name

    img_id = None
    for k, im in img_by_id.items():
        if im.get("file_name") == img_name:
            img_id = int(k)
            break
    if img_id is None:
        continue

    gt_masks, h, w = get_gt_masks_for_image(img_id)

    det = infer_model.predict(str(img_path), threshold=float(BEST_CONF))
    pred_masks, pred_confs = get_pred_masks_and_conf(det)
    pred_masks = [m for m, c in zip(pred_masks, pred_confs) if c >= BEST_CONF]

    # Normalize mask shape to GT size
    norm_pred = []
    for pm in pred_masks:
        if pm.shape != (h, w):
            pm = cv2.resize(pm.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        norm_pred.append((pm > 0).astype(np.uint8))
    pred_masks = norm_pred

    gt_union = np.zeros((h, w), dtype=np.uint8)
    for gm in gt_masks:
        gt_union = np.logical_or(gt_union, gm).astype(np.uint8)

    pred_union = np.zeros((h, w), dtype=np.uint8)
    for pm in pred_masks:
        pred_union = np.logical_or(pred_union, pm).astype(np.uint8)

    iou = iou_binary(gt_union, pred_union)
    per_image_rows.append({"image": img_name, "iou_union": iou, "n_gt": len(gt_masks), "n_pred": len(pred_masks)})

    # Render visualization
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    if img_rgb.shape[:2] != (h, w):
        img_rgb = cv2.resize(img_rgb, (w, h), interpolation=cv2.INTER_LINEAR)

    overlay = img_rgb.copy()
    overlay[gt_union == 1] = (overlay[gt_union == 1] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
    overlay[pred_union == 1] = (overlay[pred_union == 1] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
    overlap = np.logical_and(gt_union, pred_union)
    overlay[overlap] = (overlay[overlap] * 0.5 + np.array([255, 255, 0]) * 0.5).astype(np.uint8)

    diff = np.zeros((h, w, 3), dtype=np.uint8)
    diff[gt_union == 1] = [0, 200, 0]
    diff[pred_union == 1] = [200, 0, 0]
    diff[overlap] = [255, 255, 0]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f"{img_name} | IoU={iou:.3f} | GT={len(gt_masks)} | Pred={len(pred_masks)}", fontsize=12)
    axes[0].imshow(img_rgb); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(overlay); axes[1].set_title("GT=green  AI=red  Overlap=yellow"); axes[1].axis("off")
    axes[2].imshow(diff); axes[2].set_title("Difference map"); axes[2].axis("off")
    plt.tight_layout()
    plt.savefig(VIZ_DIR / f"iou_viz_{Path(img_name).stem}.png", dpi=140, bbox_inches="tight")
    plt.close(fig)

    print(f"{img_name:45s} IoU={iou:.4f}")

per_image_df = pd.DataFrame(per_image_rows).sort_values("iou_union")
PER_IMAGE_CSV = OUTPUT_DIR / "validation_per_image_iou_rfdetr.csv"
per_image_df.to_csv(PER_IMAGE_CSV, index=False)

print("\nSaved:")
print("-", VIZ_DIR)
print("-", PER_IMAGE_CSV)
print(f"Mean per-image union IoU: {per_image_df['iou_union'].mean():.4f}")
display(per_image_df)

ROSILHA-M-G_005_jpg.rf.afae2fb85329c0fe56679d51ca1c64fd.jpg IoU=0.8687
ZAZA-G_010_jpg.rf.cd14098445a0a54c10014aaa7d8f975c.jpg IoU=0.9394
ZAINA-679-B_010_jpg.rf.4a3c6644c7fdcdc349eacebe9a97fbd7.jpg IoU=0.9257
TP-B_001_jpg.rf.476ac40460a2659ffbad7973abf99d86.jpg IoU=0.9495
ROSILHA-M-G_017_jpg.rf.85fc7b8a7b63a2cbb64907a1c82d5d51.jpg IoU=0.9321
TORDILHA-B_009_jpg.rf.5902113a7929ec9592dc2b09471de461.jpg IoU=0.8935
ROSILHA-M-D_009_jpg.rf.b52035c8fe5fd87e1eed76dc4bbe6a14.jpg IoU=0.9503
TORDILHA-D_006_jpg.rf.d879d44ef1add0b327aa39f6650afc47.jpg IoU=0.7485
ZAINA-679-B_011_jpg.rf.27d46a055eff6f5810ec4c5c9429b994.jpg IoU=0.9628
ZAZA-B_008_jpg.rf.92ad451b5359143319a46e599a0239d1.jpg IoU=0.8894
ROSILHA-M-G_001_jpg.rf.3dab74a9f6c203f6aca8afc07a353bd2.jpg IoU=0.8873
TORDILHA-B_010_jpg.rf.698124083a3806b10e86f6304c2b5ce1.jpg IoU=0.9406
TOSTADA-G_011_jpg.rf.c0b219a9a6742c565eddfcdb51541b7d.jpg IoU=0.9488
ROSILHA-M-B_003_jpg.rf.863a24cebe30a134937348de5cbcaa87.jpg IoU=0.9064
ZAZA-G_012_jpg.rf.b65fca463e

,image,iou_union,n_gt,n_pred
21,ROSILHA-M-B_009_jpg.rf.d2aa8881a0b4f54b4bdb893...,0.666077,44,38
25,TORDILHA-D_001_jpg.rf.36e39c15aad81283f9bcc0f8...,0.747669,24,23
7,TORDILHA-D_006_jpg.rf.d879d44ef1add0b327aa39f6...,0.748498,35,34
15,TORDILHA-G_009_jpg.rf.b279af9f6892d54790621c0f...,0.766863,37,36
16,TP-G_010_jpg.rf.5c8b394cdec2d68bae00b4eb6679ec...,0.811115,37,34
17,TOSTADA-B_006_jpg.rf.fd9a20fd791843647a700cdeb...,0.841324,24,25
24,TOSTADA-B_003_jpg.rf.ef070508fe8cf028db79cd039...,0.847116,26,27
0,ROSILHA-M-G_005_jpg.rf.afae2fb85329c0fe56679d5...,0.868735,13,12
10,ROSILHA-M-G_001_jpg.rf.3dab74a9f6c203f6aca8afc...,0.887320,18,19
9,ZAZA-B_008_jpg.rf.92ad451b5359143319a46e599a02...,0.889363,28,26


In [4]:
# 3) Per-instance TP/FP/FN report with area error stats (YOLO-like)

# Corrected 2026-09-07: anisotropic 4140x3096 -> 640x640 stretch; correction = 3096/4140 = 0.747826
AREA_FACTOR = (50.0 / 72.0) ** 2 * (3096.0 / 4140.0)  # same conversion used in YOLO notebooks
IOU_THRESHOLD = 0.5

rows = []

for img_path in valid_img_paths:
    img_name = img_path.name

    img_id = None
    for k, im in img_by_id.items():
        if im.get("file_name") == img_name:
            img_id = int(k)
            break
    if img_id is None:
        continue

    gt_masks, h, w = get_gt_masks_for_image(img_id)

    det = infer_model.predict(str(img_path), threshold=float(BEST_CONF))
    pred_masks, pred_confs = get_pred_masks_and_conf(det)
    pred_masks = [m for m, c in zip(pred_masks, pred_confs) if c >= BEST_CONF]

    norm_pred = []
    for pm in pred_masks:
        if pm.shape != (h, w):
            pm = cv2.resize(pm.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        norm_pred.append((pm > 0).astype(np.uint8))
    pred_masks = norm_pred

    n_p, n_g = len(pred_masks), len(gt_masks)
    mat = np.zeros((n_p, n_g), dtype=np.float32)
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            mat[i, j] = iou_binary(pm, gm)

    matched_p, matched_g = set(), set()
    if n_p > 0 and n_g > 0:
        order = np.argsort(-mat, axis=None)
        pi, gi = np.unravel_index(order, mat.shape)
        for p_idx, g_idx in zip(pi, gi):
            if p_idx in matched_p or g_idx in matched_g:
                continue
            iou_val = float(mat[p_idx, g_idx])
            if iou_val < IOU_THRESHOLD:
                break

            p_area = int(pred_masks[p_idx].sum())
            g_area = int(gt_masks[g_idx].sum())
            area_err_pct = ((p_area - g_area) / g_area * 100.0) if g_area > 0 else 0.0

            rows.append(
                {
                    "Image": img_name,
                    "Type": "TP",
                    "IoU": round(iou_val, 4),
                    "GT_Area_um2": round(g_area * AREA_FACTOR, 2),
                    "AI_Area_um2": round(p_area * AREA_FACTOR, 2),
                    "Area_Err_%": round(area_err_pct, 2),
                    "Abs_Area_Err_%": round(abs(area_err_pct), 2),
                }
            )
            matched_p.add(p_idx)
            matched_g.add(g_idx)

    # False positives
    for p_idx in range(n_p):
        if p_idx in matched_p:
            continue
        p_area = int(pred_masks[p_idx].sum())
        rows.append(
            {
                "Image": img_name,
                "Type": "FP",
                "IoU": 0.0,
                "GT_Area_um2": 0.0,
                "AI_Area_um2": round(p_area * AREA_FACTOR, 2),
                "Area_Err_%": np.nan,
                "Abs_Area_Err_%": np.nan,
            }
        )

    # False negatives
    for g_idx in range(n_g):
        if g_idx in matched_g:
            continue
        g_area = int(gt_masks[g_idx].sum())
        rows.append(
            {
                "Image": img_name,
                "Type": "FN",
                "IoU": 0.0,
                "GT_Area_um2": round(g_area * AREA_FACTOR, 2),
                "AI_Area_um2": 0.0,
                "Area_Err_%": np.nan,
                "Abs_Area_Err_%": np.nan,
            }
        )

report_df = pd.DataFrame(rows)

n_tp = int((report_df["Type"] == "TP").sum())
n_fp = int((report_df["Type"] == "FP").sum())
n_fn = int((report_df["Type"] == "FN").sum())

precision = n_tp / (n_tp + n_fp) if (n_tp + n_fp) else 0.0
recall = n_tp / (n_tp + n_fn) if (n_tp + n_fn) else 0.0

tp_df = report_df[report_df["Type"] == "TP"]
mean_iou = float(tp_df["IoU"].mean()) if len(tp_df) else 0.0
mean_gt = float(tp_df["GT_Area_um2"].mean()) if len(tp_df) else 0.0
mean_ai = float(tp_df["AI_Area_um2"].mean()) if len(tp_df) else 0.0
mean_err = float(tp_df["Area_Err_%"].mean()) if len(tp_df) else 0.0
mean_abs_err = float(tp_df["Abs_Area_Err_%"].mean()) if len(tp_df) else 0.0

REPORT_CSV = OUTPUT_DIR / "placenta_instance_report_rfdetr.csv"
report_df.to_csv(REPORT_CSV, index=False)

print("\n" + "=" * 55)
print("PER-INSTANCE REPORT (RF-DETR)")
print("=" * 55)
print(f"True Positives  (matched):      {n_tp}")
print(f"False Positives (extra preds):  {n_fp}")
print(f"False Negatives (missed GT):    {n_fn}")
print(f"Precision:                      {precision:.4f}")
print(f"Recall:                         {recall:.4f}")
print("-" * 55)
print(f"Mean IoU        (TP only):      {mean_iou:.4f}")
print(f"Mean GT Area    (TP only):      {mean_gt:.2f} µm²")
print(f"Mean AI Area    (TP only):      {mean_ai:.2f} µm²")
print(f"Mean Area Error (TP only):      {mean_err:.2f}%")
print(f"Mean Abs Error  (TP only):      {mean_abs_err:.2f}%")
print("=" * 55)
print(f"Report saved to: {REPORT_CSV}")


PER-INSTANCE REPORT (RF-DETR)
True Positives  (matched):      750
False Positives (extra preds):  73
False Negatives (missed GT):    102
Precision:                      0.9113
Recall:                         0.8803
-------------------------------------------------------
Mean IoU        (TP only):      0.9020
Mean GT Area    (TP only):      2380.92 µm²
Mean AI Area    (TP only):      2303.41 µm²
Mean Area Error (TP only):      -2.88%
Mean Abs Error  (TP only):      6.76%
Report saved to: D:\projeto_placentas_clayton\dev\projeto-placentas\v2_rf_detr\runs\seg_small_v1\placenta_instance_report_rfdetr.csv
